In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd



# Basic debugging functions
def print_tensor_details(tensor):
    print('instance before debug: ', tensor)
    print('shape: ', tensor.shape)
    print('size(): ', tensor.size())
    print('device: ', tensor.device)
    print('dtype: ', tensor.dtype)
    print('grad: ', tensor.grad)
    print('grad_fn: ', tensor.grad_fn)
    # print('Tensor Row/Col', tensor[:, :])
    # print('Tensor Col-2nd', tensor[:, 1])
    # print('Tensor Row-2nd', tensor[1, :])
    print('instance after debug: ', tensor)
    pass


## The Math Behind the Code
You are calculating a simple multiplication:
$$z = 2x^3 + 3y^2 \cdot x$$

In backpropagation, we want to find out how much a small change in $x$ or $y$ affects $z$. These are the partial derivatives.

By the power rule of calculus: The partial derivative of $z$ with respect to $x$ is just $6x^2 + 3y^2$.
$$\frac{\partial z}{\partial x} = 6x^2 + 3y^2$$

The partial derivative of $z$ with respect to $y$ is just $6y \cdot x$.
$$\frac{\partial z}{\partial y} = 6y \cdot x$$

Given your inputs ($x = 2$, $y = 5$):
$$\frac{\partial z}{\partial x} = 99$$
$$\frac{\partial z}{\partial y} = 60$$

In [38]:
x = 2
y = 5
'''
Device:
    one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu
'''
tensor_x = torch.tensor(
    data=x,
    dtype=torch.float32,
    device='cpu',
    requires_grad=True
)
tensor_x.reshape((1,1))

tensor_y = torch.tensor(
    data=y,
    dtype=torch.float32,
    device='cpu',
    requires_grad=True
)
tensor_y.reshape((1,1))

tensor_z = 2*(tensor_x**3) + 3*tensor_x * tensor_y**2

# Or, alt:

# tensor_z = torch.multiply(tensor_x, tensor_y)
# Need to retain the .grad for non-leaf nodes
tensor_z.retain_grad()

In [39]:
print(tensor_x)
print(tensor_x.data)
print_tensor_details(tensor_x)
print('---------')
print(tensor_y)
print(tensor_y.data)
print_tensor_details(tensor_y)
print('---------')
print(tensor_z)
print(tensor_z.data)
print_tensor_details(tensor_z)
print('---------')

tensor(2., requires_grad=True)
tensor(2.)
instance before debug:  tensor(2., requires_grad=True)
shape:  torch.Size([])
size():  torch.Size([])
device:  cpu
dtype:  torch.float32
grad:  None
grad_fn:  None
instance after debug:  tensor(2., requires_grad=True)
---------
tensor(5., requires_grad=True)
tensor(5.)
instance before debug:  tensor(5., requires_grad=True)
shape:  torch.Size([])
size():  torch.Size([])
device:  cpu
dtype:  torch.float32
grad:  None
grad_fn:  None
instance after debug:  tensor(5., requires_grad=True)
---------
tensor(166., grad_fn=<AddBackward0>)
tensor(166.)
instance before debug:  tensor(166., grad_fn=<AddBackward0>)
shape:  torch.Size([])
size():  torch.Size([])
device:  cpu
dtype:  torch.float32
grad:  None
grad_fn:  <AddBackward0 object at 0x000001AF33D66110>
instance after debug:  tensor(166., grad_fn=<AddBackward0>)
---------


#### Note on ```retain_graph=True```
retain_graph=True is required while backpropagation, if you want to execute the same cell multiple times.

Else, the graph is dropped after the first back-propagation, causing Runtime Error.

---
RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [40]:
# 3. Perform the backward pass (calculate derivatives)
# We drop create_graph and retain_graph for standard first-order gradients
tensor_z.backward(retain_graph=True)

# 4. View the effects of backpropagation
print(f"Mathematical operation: z = 2.x^3 * x.y^2 (where x={tensor_x.item()}, y={tensor_y.item()})")
print(f"Result (z): {tensor_z.item()}\n")

print("--- Gradients (Derivatives) ---")
print(f"dz/dx (Expected = 99.0) -> PyTorch computed: {tensor_x.grad.item()}")
print(f"dz/dy (Expected = 60.0) -> PyTorch computed: {tensor_y.grad.item()}")
print(f"dz/dz (Expected: 1.0)     -> PyTorch computed: {tensor_z.grad.item()}")

tensor_x.grad.zero_()
tensor_y.grad.zero_()
tensor_z.grad.zero_()

Mathematical operation: z = 2.x^3 * x.y^2 (where x=2.0, y=5.0)
Result (z): 166.0

--- Gradients (Derivatives) ---
dz/dx (Expected = 99.0) -> PyTorch computed: 99.0
dz/dy (Expected = 60.0) -> PyTorch computed: 60.0
dz/dz (Expected: 1.0)     -> PyTorch computed: 1.0


tensor(0.)

#### For Repeated calculations of ```First Derivate```
You need to recreate the tensor within the loop again-and-again

In [41]:
for epoch in range(0,1000):

    tensor_z = 2*(tensor_x**3) + 3*tensor_x * tensor_y**2
    tensor_z.retain_grad()

    print("\n\n--- Next X, Y, Z ---")
    print(f"X: {tensor_x.item()}")
    print(f"Y: {tensor_y.item()}")
    print(f"Z: {tensor_z.item()}")

    tensor_z.backward()

    print(f"Mathematical operation: z = 2.x^3 * x.y^2 "
          f"(where x={tensor_x.item()}, y={tensor_y.item()})")
    print(f"Result (z): {tensor_z.item()}\n")

    print("--- Gradients (Derivatives) ---")
    print(f"dz/dx (Expected = 99.0) -> PyTorch computed: {tensor_x.grad.item()}")
    print(f"dz/dy (Expected = 60.0) -> PyTorch computed: {tensor_y.grad.item()}")
    print(f"dz/dz (Expected: 1.0)     -> PyTorch computed: {tensor_z.grad.item()}")

    with torch.no_grad():
        tensor_x -= 0.1 * tensor_x.grad
        tensor_y -= 0.1 * tensor_y.grad

    tensor_x.grad.zero_()
    tensor_y.grad.zero_()
    tensor_z.grad.zero_()



--- Next X, Y, Z ---
X: 2.0
Y: 5.0
Z: 166.0
Mathematical operation: z = 2.x^3 * x.y^2 (where x=2.0, y=5.0)
Result (z): 166.0

--- Gradients (Derivatives) ---
dz/dx (Expected = 99.0) -> PyTorch computed: 99.0
dz/dy (Expected = 60.0) -> PyTorch computed: 60.0
dz/dz (Expected: 1.0)     -> PyTorch computed: 1.0


--- Next X, Y, Z ---
X: -7.90000057220459
Y: -1.0
Z: -1009.7781982421875
Mathematical operation: z = 2.x^3 * x.y^2 (where x=-7.90000057220459, y=-1.0)
Result (z): -1009.7781982421875

--- Gradients (Derivatives) ---
dz/dx (Expected = 99.0) -> PyTorch computed: 377.4600524902344
dz/dy (Expected = 60.0) -> PyTorch computed: 47.400001525878906
dz/dz (Expected: 1.0)     -> PyTorch computed: 1.0


--- Next X, Y, Z ---
X: -45.6460075378418
Y: -5.740000247955322
Z: -194724.0
Mathematical operation: z = 2.x^3 * x.y^2 (where x=-45.6460075378418, y=-5.740000247955322)
Result (z): -194724.0

--- Gradients (Derivatives) ---
dz/dx (Expected = 99.0) -> PyTorch computed: 12600.19140625
dz/dy (

## Second Order Derivative Calculations

To calculate second-order (or higher) derivatives, we use torch.autograd.grad. This is exactly the function the PyTorch warning from your previous step recommended.

By using autograd.grad instead of .backward(), you gain finer control over the computation graph and avoid the memory leak issues associated with attaching gradients directly to the .grad attributes of your parameters.

### The Math
Let's use a function where the second derivative isn't zero.

We will calculate the derivatives of:
$$z = x^3$$
Using the power rule, the math looks like this:
- First derivative: $\frac{\partial z}{\partial x} = 3x^2$
- Second derivative: $\frac{\partial^2 z}{\partial x^2} = 6x$

If we set $x = 2$:$z = 2^3 = 8$
- First derivative $= 3(2)^2 = 12$
- Second derivative $= 6(2) = 12$

In [42]:
# 1. Define the input tensor
Tx = torch.tensor(float(x), requires_grad=True)

# 2. Define the forward pass (the equation)
Tz = Tx ** 3

# 3. Calculate the first derivative (dz/dx)
# We use create_graph=True so PyTorch tracks the math used to create this gradient.
# Note: autograd.grad returns a tuple, so we use [0] to extract the actual tensor.
grad_1 = torch.autograd.grad(outputs=Tz, inputs=Tx, create_graph=True)[0]

# 4. Calculate the second derivative d(dz/dx)/dx
# We are differentiating grad_1 with respect to x.
# No need for create_graph=True here unless you want a third derivative!
grad_2 = torch.autograd.grad(outputs=grad_1, inputs=Tx, create_graph=True)[0]

# 5. Print the results
print(f"Function: z = x^3 (where x = {Tx.item()})\n")
print(f"z      (Expected: 8.0)  -> Computed: {Tz.item()}")
print(f"dz/dx  (Expected: 12.0) -> Computed: {grad_1.item()}")
print(f"d2z/dx2 (Expected: 12.0) -> Computed: {grad_2.item()}")

Function: z = x^3 (where x = 2.0)

z      (Expected: 8.0)  -> Computed: 8.0
dz/dx  (Expected: 12.0) -> Computed: 12.0
d2z/dx2 (Expected: 12.0) -> Computed: 12.0


In [47]:
# No need for create_graph=True here unless you want a fourth derivative!
## However, retain_graph=True preserves the previous graph,
## so you could execute it multiple times!
grad_3 = torch.autograd.grad(outputs=grad_2, inputs=Tx,
                             retain_graph=True)[0]
print(f"Triple/3rd-order Derivative: d3z/dx3 (Expected: 6.0) -> Computed: {grad_3.item()}")

Triple/3rd-order Derivative: d3z/dx3 (Expected: 6.0) -> Computed: 6.0


## Other Basic Tensor creations

In [44]:
# Random initializations / seeding
## 2D
tensor_rand_1_1 = torch.randn(1, 1, requires_grad=True)
## 1D
tensor_rand_1 = torch.randn(1, requires_grad=True)
## 3D
tensor_rand_1_1_1 = torch.randn(1, 1, 1, requires_grad=True)

# Now print those:
print(tensor_rand_1_1)
print(tensor_rand_1)
print(tensor_rand_1_1_1)

tensor([[-0.5871]], requires_grad=True)
tensor([1.1456], requires_grad=True)
tensor([[[0.5908]]], requires_grad=True)
